# **ETL BRONZE PROCESS**

In [0]:
%pip install pypdf

In [0]:
import os
import base64
from databricks.sdk import WorkspaceClient
from openai import OpenAI
from pypdf import PdfReader
import json
from pyspark.sql.functions import lit, current_timestamp

prompt= """

1. FORMATO DE SALIDA (JSON)
La respuesta debe ser únicamente JSON válido, con la siguiente estructura exacta:

	json
	[
	 {
		"nombre_mercado": "null",
		"nombre_producto": "null",
		"unidad_medida": "null",
		"inicio_fecha_precios": "null",
		"fin_fecha_precios": "null",
		"precio_inicio": "null",
		"precio_fin": "null",
		"error": "null"
	 }
	]
Regla crítica: Todos los valores deben ir como strings (entre comillas dobles), incluyendo "null" cuando no haya dato.

2. REGLAS DE EXTRACCIÓN DE CAMPOS
2.1 nombre_mercado
	Tomar el texto que está dentro del rectángulo amarillo (título del mercado/feria/supermercado).

	Normalizar contra la lista oficial de mercados comunes (minúsculas, con tildes correctas).

	Si el texto no coincide con la lista, mantener el texto tal cual se leyó.

	Si no se puede leer, colocar "null" y reportar en "error".

	Lista oficial de mercados normalizados:

	- mercado zonal belén, comayagüela
	- ahorro ferias del pueblo
	- feria del agricultor y el artesano
	- supermercados tegucigalpa
	- supermercados san pedro sula
	- mercado el rápido san pedro sula
	- feria agropecuaria de villanueva
	- mercado san isidro, las americas y colón

2.2 nombre_producto
	Usar el catálogo estándar de productos según la posición de la fila en la tabla.
	Si la posición no coincide con el catálogo, intentar leer el texto directamente.
	Si no se puede leer, colocar "null" y reportar en "error".

2.3 unidad_medida
	Usar el catálogo estándar de unidades según la posición de la fila.
	Si no se puede determinar, colocar "null" y reportar en "error".

2.4 inicio_fecha_precios y fin_fecha_precios
	Buscar el texto "( Semana del ...)"* o "Semana del ..." en la parte superior de las columnas.
	Dividirlo en dos fechas con formato DD-MM-AAAA:
	inicio_fecha_precios: primer valor de la semana
	fin_fecha_precios: segundo valor de la semana

2.5 precio_inicio
	Tomar el valor de la primera columna de "Precios L" (o "Precio L. [fecha inicio]").

2.6 precio_fin
	Tomar el valor de la segunda columna de "Precios L" (o "Precio L. [fecha fin]").

2.7 error
	Si no hay problemas: "null"
	Si hay problemas de lectura: indicar número de página + nombre del documento.
	Formato exacto: "pag X - nombre_del_archivo.pdf"

	3. REGLAS DE PROCESAMIENTO DE PÁGINAS Y TABLAS
	Procesar todas las páginas del archivo, sin excepción, desde la página 1 hasta la última.
	Incluir todas las tablas que aparezcan en cada página, aunque estén parcialmente cortadas, borrosas o incompletas.
	Si una tabla no se puede leer en absoluto:
	Crear una sola entrada JSON para esa tabla.
	Poner "null" en todos los campos.
	Indicar en "error" el número de página + nombre del documento.
	Si una tabla se puede leer parcialmente:
	Incluir una entrada JSON por cada fila que se pueda leer.
	En las filas donde falte algún valor, poner "null" en ese campo.
	En el campo "error" de esa fila, indicar el número de página + nombre del documento.
	Nunca omitir una tabla por estar borrosa, cortada o con texto corrupto.
	Si una página contiene solo ruido o texto sin estructura de tabla:
	No se genera entrada de tabla.
	Se debe reportar con una entrada JSON con todos los campos "null" y en "error" el número de página + nombre del documento.
	No agrupar ni resumir: cada fila de cada tabla debe ser un objeto JSON independiente.
	Si no se puede leer un valor de una fila de la tabla, colocar "null" en ese campo (no omitirlo) y reportar en "error" el número de página + nombre del documento.

4. CATÁLOGO ESTÁNDAR DE PRODUCTOS Y UNIDADES
	Usar este catálogo según la posición de la fila en la tabla (1 a 30):

	#	Producto	Unidad
	1	Tajo de Res	libra
	2	Costilla de Res Regular	libra
	3	Costilla de Cerdo Regular	libra
	4	Pollo Entero Congelado sin Menudos	libra
	5	Pescado Blanco	libra
	6	Leche Integra en Polvo	360 g
	7	Leche pasteurizada fluida en bolsa	0.946l
	8	Mantequilla	libra
	9	Queso Blanco Fresco	libra
	10	Huevo Mediano	cartón
	11	Cebolla Amarilla	libra
	12	Tomate Pera	libra
	13	Papas	libra
	14	Yuca	libra
	15	Repollo	libra
	16	Plátano Maduro	unidad
	17	Naranja Dulce	unidad
	18	Banano Fresco Maduro	unidad
	19	Frijol Rojo a Granel	libra
	20	Arroz Clasificado a Granel	libra
	21	Espagueti	200 g
	22	Azúcar Blanca	libra
	23	Café Molido	16 oz
	24	Salsa de Tomate	400 g
	25	Aceite Vegetal	443 ml
	26	Manteca Comestible de Origen Vegetal	libra
	27	Sal común de Mesa Yodada	227 g
	28	Tortilla de Maíz	unidad
	29	Pan Molde Blanco	540 g
	30	Jugo de Naranja	500ml
	Regla adicional: Cuando una tabla no tenga exactamente 30 filas, usar la posición de cada fila para asignar el nombre y unidad correctos según este catálogo, en lugar de depender del texto ilegible.

5. CORRECCIONES COMUNES DE LECTURA (APLICAR AUTOMÁTICAMENTE)
5.1 Productos
	Texto ilegible	Corrección
	Cotilla	Costilla
	Mantegulla	Mantequilla
	Horro Mediano / Horno Mediano / Hervo Mediano	Huevo Mediano
	Pilatano Maduro / Píñano Maduro	Plátano Maduro
	Fitjol Rojo / Frijol Rojo a Granad	Frijol Rojo a Granel
	Acerle Vegetal	Aceite Vegetal
	Leche Pasteurizada India en Bolos	Leche pasteurizada fluida en bolsa
	Leche de Menudo	Manteca Comestible de Origen Vegetal
	Tomate Pecа / Teca	Tomate Pera / Yuca
	Anicar Blanca	Azúcar Blanca
	Pan Molido Blanco	Pan Molde Blanco
	Jugo de Naranja (Bolsa 500 ml)	Jugo de Naranja (500ml)

5.2 Mercados
	Texto ilegible	Corrección
	Mercado Zonal Belén, Comayaguela	mercado zonal belén, comayagüela
	Ahorro Ferias del Pueblo / Ahorro Feria del Pueblo	ahorro ferias del pueblo
	Feria del Agricultor y El Artesano	feria del agricultor y el artesano
	Supermercados Tegucigalpa / Supermercados Tequiegala	supermercados tegucigalpa
	Supermercado San Pedro Sula / Supermercados San Pedro Sula	supermercados san pedro sula
	Mercado El Rápido San Pedro Sula / Mercado El Rapido San Pedro Sula	mercado el rápido san pedro sula
	Feria Agropecuaria de Villanueva / Feria Agropecuaria Villanueva	feria agropecuaria de villanueva
	Mercados San Isidro, las Americas y Colón	mercado san isidro, las americas y colón
	
6. CHECKLIST FINAL ANTES DE ENTREGAR
	-¿Se procesaron todas las páginas del PDF?
	-¿Se incluyó al menos una entrada JSON por cada tabla encontrada?
	-¿Todos los valores están entre comillas dobles (strings)?
	-¿Los campos sin dato tienen "null" y su "error" correspondiente?
	-¿Los nombres de mercado están normalizados según la lista oficial?
	-¿Los nombres de producto y unidades siguen el catálogo estándar?
	-¿Las fechas están en formato DD-MM-AAAA?
	-¿El campo "error" usa el formato "pag X - nombre_del_archivo.pdf"?
	-¿No se agrupó ni resumió ninguna fila?
	-¿El JSON es válido y parseable?

"""

In [0]:


wkc = WorkspaceClient()


def api_execute(file_name,prompt):
    #1 Read File Properties
    reader = PdfReader(file_name)
    
    file_content = ""
    for page in reader.pages:
        text = page.extract_text()
        if text:
            file_content += text + "\n"

  
    # 2. Inicializa el cliente e ingresa tu API key directamente aquí
    client = OpenAI(
        api_key="sk-9a2fa846dcf2449a9915032c6f12ac6d",  # <-- AQUÍ COLOCAS TU API KEY
        base_url="https://api.deepseek.com"
    )

    print("Llegue aqui al paso 2")
    
    # 3. Envía la solicitud con tu prompt y el contenido del archivo
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {
                "role": "user",
                "content": 
                    f"{prompt}\n\n{file_content}",
            },
        ],
    )       

    print(response.choices[0].message.content)

In [0]:
#Execute the PDF reading for OCR in Deepseek and save the results to a dataframe

def readFileContainer(prompt):
    # List files in a workspace directory
    for file in wkc.workspace.list(
        "/Workspace/Users/jarodriguezv91@gmail.com/TESTING/DOCS/"
    ):
        file_name = file.path
        query = f"SELECT 1 FROM bootcamp.bronze.docs_name WHERE docs_name = '{file_name}' LIMIT 1"

        df = spark.sql(query)
      
        #api_execute(file_name, prompt)


        if df.isEmpty():
            query = f"INSERT INTO bootcamp.bronze.docs_name(docs_name) VALUES ('{file_name}')"
            df = spark.sql(query)

readFileContainer(prompt)

In [0]:
query_json= [
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Tajo de Res",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "95.00",
    "precio_fin": "95.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Costilla de Res Regular",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "80.00",
    "precio_fin": "85.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Costilla de Cerdo Regular",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "73.33",
    "precio_fin": "70.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Pollo Entero Congelado sin Menudos",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "32.00",
    "precio_fin": "32.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Pescado Blanco",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "38.33",
    "precio_fin": "35.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Leche Integra en Polvo",
    "unidad_medida": "360 g",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "125.00",
    "precio_fin": "124.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Leche pasteurizada fluida en bolsa",
    "unidad_medida": "0.946l",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "30.00",
    "precio_fin": "30.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Mantequilla",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "32.00",
    "precio_fin": "32.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Queso Blanco Fresco",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "95.33",
    "precio_fin": "95.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Huevo Mediano",
    "unidad_medida": "cartón",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "90.00",
    "precio_fin": "85.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Cebolla Amarilla",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "26.67",
    "precio_fin": "20.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Tomate Pera",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "15.00",
    "precio_fin": "20.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Papas",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "15.00",
    "precio_fin": "12.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Yuca",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "10.00",
    "precio_fin": "12.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Repollo",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "6.00",
    "precio_fin": "6.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Plátano Maduro",
    "unidad_medida": "unidad",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "10.00",
    "precio_fin": "10.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Naranja Dulce",
    "unidad_medida": "unidad",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "5.00",
    "precio_fin": "5.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Banano Fresco Maduro",
    "unidad_medida": "unidad",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "5.00",
    "precio_fin": "5.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Frijol Rojo a Granel",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "22.00",
    "precio_fin": "22.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Arroz Clasificado a Granel",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "13.33",
    "precio_fin": "14.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Espagueti",
    "unidad_medida": "200 g",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "10.00",
    "precio_fin": "12.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Azúcar Blanca",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "14.00",
    "precio_fin": "15.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Café Molido",
    "unidad_medida": "16 oz",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "120.00",
    "precio_fin": "120.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Salsa de Tomate",
    "unidad_medida": "400 g",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "44.33",
    "precio_fin": "44.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Aceite Vegetal",
    "unidad_medida": "443 ml",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "37.33",
    "precio_fin": "35.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Manteca Comestible de Origen Vegetal",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "38.33",
    "precio_fin": "35.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Sal común de Mesa Yodada",
    "unidad_medida": "227 g",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "null",
    "precio_fin": "null",
    "error": "pag 1 - 5_mayo_2026.pdf"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Tortilla de Maíz",
    "unidad_medida": "unidad",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "1.00",
    "precio_fin": "1.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Pan Molde Blanco",
    "unidad_medida": "540 g",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "56.00",
    "precio_fin": "56.00",
    "error": "null"
  },
  {
    "nombre_mercado": "mercado zonal belén, comayagüela",
    "nombre_producto": "Jugo de Naranja",
    "unidad_medida": "500ml",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "15.33",
    "precio_fin": "14.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Tajo de Res",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "90.00",
    "precio_fin": "90.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Costilla de Res Regular",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "72.00",
    "precio_fin": "75.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Costilla de Cerdo Regular",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "70.00",
    "precio_fin": "70.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Pollo Entero Congelado sin Menudos",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "27.50",
    "precio_fin": "26.50",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Pescado Blanco",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "25.00",
    "precio_fin": "30.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Leche Integra en Polvo",
    "unidad_medida": "360 g",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "35.00",
    "precio_fin": "125.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Leche pasteurizada fluida en bolsa",
    "unidad_medida": "0.946l",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "30.00",
    "precio_fin": "30.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Mantequilla",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "38.00",
    "precio_fin": "38.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Queso Blanco Fresco",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "38.00",
    "precio_fin": "38.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Huevo Mediano",
    "unidad_medida": "cartón",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "78.00",
    "precio_fin": "78.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Cebolla Amarilla",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "15.00",
    "precio_fin": "15.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Tomate Pera",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "12.00",
    "precio_fin": "10.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Papas",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "12.00",
    "precio_fin": "13.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Yuca",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "10.00",
    "precio_fin": "10.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Repollo",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "6.00",
    "precio_fin": "6.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Plátano Maduro",
    "unidad_medida": "unidad",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "10.00",
    "precio_fin": "10.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Naranja Dulce",
    "unidad_medida": "unidad",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "5.00",
    "precio_fin": "5.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Banano Fresco Maduro",
    "unidad_medida": "unidad",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "3.33",
    "precio_fin": "3.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Frijol Rojo a Granel",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "20.00",
    "precio_fin": "20.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Arroz Clasificado a Granel",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "14.00",
    "precio_fin": "14.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Espagueti",
    "unidad_medida": "200 g",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "11.00",
    "precio_fin": "11.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Azúcar Blanca",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "14.00",
    "precio_fin": "14.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Café Molido",
    "unidad_medida": "16 oz",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "115.00",
    "precio_fin": "127.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Salsa de Tomate",
    "unidad_medida": "400 g",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "45.00",
    "precio_fin": "42.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Aceite Vegetal",
    "unidad_medida": "443 ml",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "40.00",
    "precio_fin": "40.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Manteca Comestible de Origen Vegetal",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "30.00",
    "precio_fin": "30.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Sal común de Mesa Yodada",
    "unidad_medida": "227 g",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "3.50",
    "precio_fin": "3.50",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Tortilla de Maíz",
    "unidad_medida": "unidad",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "3.50",
    "precio_fin": "3.50",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Pan Molde Blanco",
    "unidad_medida": "540 g",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "55.00",
    "precio_fin": "50.00",
    "error": "null"
  },
  {
    "nombre_mercado": "ahorro ferias del pueblo",
    "nombre_producto": "Jugo de Naranja",
    "unidad_medida": "500ml",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "16.00",
    "precio_fin": "14.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Tajo de Res",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "94.00",
    "precio_fin": "94.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Costilla de Res Regular",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "78.00",
    "precio_fin": "78.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Costilla de Cerdo Regular",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "68.00",
    "precio_fin": "68.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Pollo Entero Congelado sin Menudos",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "26.50",
    "precio_fin": "26.50",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Pescado Blanco",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "40.00",
    "precio_fin": "35.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Leche Integra en Polvo",
    "unidad_medida": "360 g",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "120.00",
    "precio_fin": "125.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Leche pasteurizada fluida en bolsa",
    "unidad_medida": "0.946l",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "27.00",
    "precio_fin": "27.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Mantequilla",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "32.00",
    "precio_fin": "32.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Queso Blanco Fresco",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "56.00",
    "precio_fin": "56.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Huevo Mediano",
    "unidad_medida": "cartón",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "78.00",
    "precio_fin": "78.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Cebolla Amarilla",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "12.00",
    "precio_fin": "12.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Tomate Pera",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "15.00",
    "precio_fin": "15.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Papas",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "12.00",
    "precio_fin": "10.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Yuca",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "10.00",
    "precio_fin": "10.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Repollo",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "7.00",
    "precio_fin": "6.25",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Plátano Maduro",
    "unidad_medida": "unidad",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "10.00",
    "precio_fin": "10.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Naranja Dulce",
    "unidad_medida": "unidad",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "5.00",
    "precio_fin": "5.50",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Banano Fresco Maduro",
    "unidad_medida": "unidad",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "2.67",
    "precio_fin": "2.67",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Frijol Rojo a Granel",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "21.50",
    "precio_fin": "21.50",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Arroz Clasificado a Granel",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "13.00",
    "precio_fin": "13.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Espagueti",
    "unidad_medida": "200 g",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "10.00",
    "precio_fin": "10.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Azúcar Blanca",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "14.00",
    "precio_fin": "15.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Café Molido",
    "unidad_medida": "16 oz",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "120.00",
    "precio_fin": "125.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Salsa de Tomate",
    "unidad_medida": "400 g",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "43.00",
    "precio_fin": "45.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Aceite Vegetal",
    "unidad_medida": "443 ml",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "36.00",
    "precio_fin": "37.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria del agricultor y el artesano",
    "nombre_producto": "Manteca Comestible de Origen Vegetal",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "04-05-2026",
    "precio_inicio": "null",
    "precio_fin": "null",
    "error": "pag 1 - 5_mayo_2026.pdf"
  },
  {
    "nombre_mercado": "null",
    "nombre_producto": "null",
    "unidad_medida": "null",
    "inicio_fecha_precios": "null",
    "fin_fecha_precios": "null",
    "precio_inicio": "null",
    "precio_fin": "null",
    "error": "pag 2 - 5_mayo_2026.pdf"
  },
  {
    "nombre_mercado": "null",
    "nombre_producto": "null",
    "unidad_medida": "null",
    "inicio_fecha_precios": "null",
    "fin_fecha_precios": "null",
    "precio_inicio": "null",
    "precio_fin": "null",
    "error": "pag 3 - 5_mayo_2026.pdf"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Tajo de Res",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "120.67",
    "precio_fin": "121.33",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Costilla de Res Regular",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "77.88",
    "precio_fin": "75.88",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Costilla de Cerdo Regular",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "71.00",
    "precio_fin": "68.88",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Pollo Entero Congelado sin Menudos",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "39.12",
    "precio_fin": "41.88",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Pescado Blanco",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "61.00",
    "precio_fin": "62.33",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Leche Integra en Polvo",
    "unidad_medida": "360 g",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "133.10",
    "precio_fin": "131.38",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Leche pasteurizada fluida en bolsa",
    "unidad_medida": "0.946l",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "36.88",
    "precio_fin": "36.12",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Mantequilla",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "66.82",
    "precio_fin": "70.66",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Queso Blanco Fresco",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "73.83",
    "precio_fin": "73.83",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Huevo Mediano",
    "unidad_medida": "cartón",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "111.07",
    "precio_fin": "109.73",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Cebolla Amarilla",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "20.27",
    "precio_fin": "20.37",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Tomate Pera",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "10.87",
    "precio_fin": "18.18",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Papas",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "24.22",
    "precio_fin": "23.82",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Yuca",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "12.05",
    "precio_fin": "12.48",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Repollo",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "10.55",
    "precio_fin": "10.20",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Plátano Maduro",
    "unidad_medida": "unidad",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "13.72",
    "precio_fin": "13.63",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Naranja Dulce",
    "unidad_medida": "unidad",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "6.90",
    "precio_fin": "6.90",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Banano Fresco Maduro",
    "unidad_medida": "unidad",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "2.88",
    "precio_fin": "3.33",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Frijol Rojo a Granel",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "30.20",
    "precio_fin": "27.42",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Arroz Clasificado a Granel",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "11.90",
    "precio_fin": "12.05",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Espagueti",
    "unidad_medida": "200 g",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "15.33",
    "precio_fin": "13.30",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Azúcar Blanca",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "13.08",
    "precio_fin": "13.34",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Café Molido",
    "unidad_medida": "16 oz",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "121.08",
    "precio_fin": "122.08",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Salsa de Tomate",
    "unidad_medida": "400 g",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "45.65",
    "precio_fin": "44.65",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Aceite Vegetal",
    "unidad_medida": "443 ml",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "40.40",
    "precio_fin": "41.00",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Manteca Comestible de Origen Vegetal",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "27.83",
    "precio_fin": "28.87",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Sal común de Mesa Yodada",
    "unidad_medida": "227 g",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "6.43",
    "precio_fin": "5.73",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Tortilla de Maíz",
    "unidad_medida": "unidad",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "1.88",
    "precio_fin": "2.04",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Pan Molde Blanco",
    "unidad_medida": "540 g",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "55.65",
    "precio_fin": "54.05",
    "error": "null"
  },
  {
    "nombre_mercado": "supermercados san pedro sula",
    "nombre_producto": "Jugo de Naranja",
    "unidad_medida": "500ml",
    "inicio_fecha_precios": "30-04-2026",
    "fin_fecha_precios": "08-05-2026",
    "precio_inicio": "17.73",
    "precio_fin": "17.32",
    "error": "null"
  },
  {
    "nombre_mercado": "null",
    "nombre_producto": "null",
    "unidad_medida": "null",
    "inicio_fecha_precios": "null",
    "fin_fecha_precios": "null",
    "precio_inicio": "null",
    "precio_fin": "null",
    "error": "pag 5 - 5_mayo_2026.pdf"
  },
  {
    "nombre_mercado": "null",
    "nombre_producto": "null",
    "unidad_medida": "null",
    "inicio_fecha_precios": "null",
    "fin_fecha_precios": "null",
    "precio_inicio": "null",
    "precio_fin": "null",
    "error": "pag 6 - 5_mayo_2026.pdf"
  },
  {
    "nombre_mercado": "null",
    "nombre_producto": "null",
    "unidad_medida": "null",
    "inicio_fecha_precios": "null",
    "fin_fecha_precios": "null",
    "precio_inicio": "null",
    "precio_fin": "null",
    "error": "pag 7 - 5_mayo_2026.pdf"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Tajo de Res",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "95.00",
    "precio_fin": "95.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Costilla de Res Regular",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "75.00",
    "precio_fin": "75.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Costilla de Cerdo Regular",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "70.00",
    "precio_fin": "65.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Pollo Entero Congelado sin Menudos",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "26.50",
    "precio_fin": "27.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Pescado Blanco",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "35.00",
    "precio_fin": "30.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Leche Integra en Polvo",
    "unidad_medida": "360 g",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "125.00",
    "precio_fin": "123.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Leche pasteurizada fluida en bolsa",
    "unidad_medida": "0.946l",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "30.00",
    "precio_fin": "30.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Mantequilla",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "28.00",
    "precio_fin": "28.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Queso Blanco Fresco",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "56.00",
    "precio_fin": "56.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Huevo Mediano",
    "unidad_medida": "cartón",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "83.00",
    "precio_fin": "83.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Cebolla Amarilla",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "15.00",
    "precio_fin": "20.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Tomate Pera",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "12.00",
    "precio_fin": "13.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Papas",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "15.00",
    "precio_fin": "17.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Yuca",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "10.00",
    "precio_fin": "12.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Repollo",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "5.00",
    "precio_fin": "5.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Plátano Maduro",
    "unidad_medida": "unidad",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "10.00",
    "precio_fin": "9.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Naranja Dulce",
    "unidad_medida": "unidad",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "6.67",
    "precio_fin": "6.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Banano Fresco Maduro",
    "unidad_medida": "unidad",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "4.00",
    "precio_fin": "3.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Frijol Rojo a Granel",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "22.00",
    "precio_fin": "22.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Arroz Clasificado a Granel",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "13.00",
    "precio_fin": "14.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Espagueti",
    "unidad_medida": "200 g",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "10.00",
    "precio_fin": "10.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Azúcar Blanca",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "15.00",
    "precio_fin": "14.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Café Molido",
    "unidad_medida": "16 oz",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "120.00",
    "precio_fin": "120.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Salsa de Tomate",
    "unidad_medida": "400 g",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "47.00",
    "precio_fin": "50.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Aceite Vegetal",
    "unidad_medida": "443 ml",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "38.00",
    "precio_fin": "40.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Manteca Comestible de Origen Vegetal",
    "unidad_medida": "libra",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "29.00",
    "precio_fin": "29.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Sal común de Mesa Yodada",
    "unidad_medida": "227 g",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "3.50",
    "precio_fin": "3.50",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Tortilla de Maíz",
    "unidad_medida": "unidad",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "1.00",
    "precio_fin": "1.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Pan Molde Blanco",
    "unidad_medida": "540 g",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "55.00",
    "precio_fin": "56.00",
    "error": "null"
  },
  {
    "nombre_mercado": "feria agropecuaria de villanueva",
    "nombre_producto": "Jugo de Naranja",
    "unidad_medida": "500ml",
    "inicio_fecha_precios": "22-05-2026",
    "fin_fecha_precios": "29-05-2026",
    "precio_inicio": "15.00",
    "precio_fin": "15.00",
    "error": "null"
  },
  {
    "nombre_mercado": "null",
    "nombre_producto": "null",
    "unidad_medida": "null",
    "inicio_fecha_precios": "null",
    "fin_fecha_precios": "null",
    "precio_inicio": "null",
    "precio_fin": "null",
    "error": "pag 9 - 5_mayo_2026.pdf"
  }
]

In [0]:
# Formating the JSON into String FORMAT

data = query_json
data_all_string= [{k: str(v) for k, v in d.items()} for d in data]

print(json.dumps(data_all_string,indent=5))

In [0]:
# Defining the column order for dataset

column_order = [
    "nombre_mercado",
    "nombre_producto",
    "unidad_medida",
    "inicio_fecha_precios",
    "fin_fecha_precios",
    "precio_inicio",
    "precio_fin",
    "error",
]


df_products = spark.createDataFrame(data_all_string)
df_product_list = df_products.select(column_order)

In [0]:
#Adding timestamp to the dataset

df_product_list_bronze = df_product_list.withColumn("ingesta_timestamp", current_timestamp())



In [0]:
#Sending the dataset to be written in the table

df_product_list_bronze.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("products.bronze.product_list_markets_bronze")

In [0]:
%sql
--Checking Bronze table data 

select * from products.bronze.product_list_markets_bronze

In [0]:
df_product_list_bronze.printSchema()